In [24]:
import pandas as pd
import numpy as np

## 1. Loading Data and initialization of *Silver Layer*

In [25]:
path_file = r'C:\Users\Maciek\Desktop\netflixdb\databases\search_logs.csv'
df_bronze = pd.read_csv(path_file, sep = ',')

df_silver = df_bronze.copy()
df_silver.head()

,search_id,user_id,search_query,search_date,results_returned,clicked_result_position,device_type,search_duration_seconds,had_typo,used_filters,location_country
0,search_000001,user_09864,classic movies,2024-03-22,20,2.0,Tablet,12.4,False,False,Canada
1,search_000002,user_08038,stand up comedy,2025-11-22,24,4.0,Tablet,63.5,True,False,USA
2,search_000003,user_02009,music documentaries,2024-10-09,86,1.0,Tablet,24.7,True,False,USA
3,search_000004,user_01083,comedy shows,2024-12-14,70,4.0,Mobile,53.7,False,False,USA
4,search_000005,user_04269,movies based on true stories,2025-01-10,48,NaN,Tablet,69.6,True,False,USA


## 2. Analyses and Exploration Nulls

In [26]:
df_silver.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 26500 entries, 0 to 26499
Data columns (total 11 columns):
 #   Column                   Non-Null Count  Dtype  
---  ------                   --------------  -----  
 0   search_id                26500 non-null  object 
 1   user_id                  26500 non-null  object 
 2   search_query             26500 non-null  object 
 3   search_date              26500 non-null  object 
 4   results_returned         26500 non-null  int64  
 5   clicked_result_position  12951 non-null  float64
 6   device_type              26500 non-null  object 
 7   search_duration_seconds  25223 non-null  float64
 8   had_typo                 26500 non-null  bool   
 9   used_filters             26500 non-null  bool   
 10  location_country         26500 non-null  object 
dtypes: bool(2), float64(2), int64(1), object(6)
memory usage: 1.9+ MB


In [27]:
df_silver.isnull().sum()

search_id                      0
user_id                        0
search_query                   0
search_date                    0
results_returned               0
clicked_result_position    13549
device_type                    0
search_duration_seconds     1277
had_typo                       0
used_filters                   0
location_country               0
dtype: int64

## 3. Handling duplicates in search_id

In [34]:
df_duplicates = df_silver[df_silver['search_id'].duplicated(keep=False)]
print(df_duplicates[['search_id', 'user_id', 'search_duration_seconds']].sort_values('search_id'))

df_silver['search_id'] = df_silver['search_id'].drop_duplicates(keep = 'first')

           search_id     user_id  search_duration_seconds
8      search_000009  user_03999                      7.0
25575  search_000009  user_03999                      7.0
23     search_000024  user_01317                     45.8
26234  search_000024  user_01317                     45.8
24     search_000025  user_03286                      NaN
...              ...         ...                      ...
24971  search_024972  user_03145                      2.4
25907  search_024974  user_06376                     16.2
24973  search_024974  user_06376                     16.2
26325  search_024983  user_05086                     10.6
24982  search_024983  user_05086                     10.6

[2958 rows x 3 columns]


In [ ]:
# Check for duplicates in 'search_id' column
print(df_silver[df_silver['search_id'].duplicated(keep=False)])

      search_id     user_id                 search_query search_date  \
25000       NaN  user_05473                documentaries  2024-11-13   
25001       NaN  user_02433  award winning documentaries  2025-05-09   
25002       NaN  user_08274                   war movies  2025-10-22   
25003       NaN  user_02456            romantic comedies  2025-06-05   
25004       NaN  user_00627  award winning documentaries  2025-06-24   
...         ...         ...                          ...         ...   
26495       NaN  user_02131              stand up comedy  2025-05-09   
26496       NaN  user_07935              thriller series  2024-03-05   
26497       NaN  user_05028               classic movies  2024-11-12   
26498       NaN  user_09780                 comedy shows  2025-06-04   
26499       NaN  user_07537              stand up comedy  2024-03-29   

       results_returned  clicked_result_position device_type  \
25000                85                      NaN      Tablet   
25001  

In [ ]:
# taking maximum existing numeric ID
numeric_ids = pd.to_numeric(
    df_silver['search_id'].str.replace(r'search_', '', regex=False),
    errors='coerce' 
) 

max_existing_id = numeric_ids.max()

if pd.isna(max_existing_id):
    max_existing_id = 0
else:
    max_existing_id = int(max_existing_id)

nan_count = df_silver['search_id'].isnull().sum()
print(nan_count)

1500


In [ ]:
# generating new numeric IDs
new_numeric_sequence = np.arange(max_existing_id + 1, max_existing_id + 1 + nan_count)

# formatting new IDs
new_formatted_ids = [f'search_{i:06d}' for i in new_numeric_sequence]

is_nan_mask = df_silver['search_id'].isna()

df_silver.loc[is_nan_mask, 'search_id'] = new_formatted_ids

print("Weryfikacja nowych ID:")
print(df_silver.loc[df_silver['search_id'].str.contains('search_')].tail())

Weryfikacja nowych ID:
           search_id     user_id     search_query search_date  \
26495  search_026496  user_02131  stand up comedy  2025-05-09   
26496  search_026497  user_07935  thriller series  2024-03-05   
26497  search_026498  user_05028   classic movies  2024-11-12   
26498  search_026499  user_09780     comedy shows  2025-06-04   
26499  search_026500  user_07537  stand up comedy  2024-03-29   

       results_returned  clicked_result_position device_type  \
26495                80                      6.0    Smart TV   
26496                36                      NaN    Smart TV   
26497                18                      4.0    Smart TV   
26498                83                      NaN      Laptop   
26499                 9                      2.0      Laptop   

       search_duration_seconds  had_typo  used_filters location_country  
26495                     25.0      True          True           Canada  
26496                     15.6     False          Tru

In [42]:
df_silver['clicked_result_position']

0        2.0
1        4.0
2        1.0
3        4.0
4        NaN
        ... 
26495    6.0
26496    NaN
26497    4.0
26498    NaN
26499    2.0
Name: clicked_result_position, Length: 26500, dtype: float64

In [43]:
df_silver['clicked_result_position']
df_silver['clicked_result_position'].unique()

array([ 2.,  4.,  1., nan,  5.,  7.,  3., 10.,  8.,  9.,  6.])

## 4. Cleaning and Transformating clicked_result_position

In [ ]:
# Calculate median value excluding NaNs
median_value = df_silver['clicked_result_position'].median()

# Create indicator column for missing values
df_silver['is_clicked_result_position_missing'] = df_silver['clicked_result_position'].isna().astype(int)
df_silver['clicked_result_position'] = df_silver['clicked_result_position'].fillna(median_value)

In [45]:
median_value1 = df_silver['search_duration_seconds'].median()

df_silver['is_search_duration_seconds_missing'] = df_silver['search_duration_seconds'].isna().astype(int)
df_silver['search_duration_seconds'] = df_silver['search_duration_seconds'].fillna(median_value1)



In [47]:
df_silver['clicked_result_position'] = df_silver['clicked_result_position'].astype(int)

In [48]:
df_silver.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 26500 entries, 0 to 26499
Data columns (total 13 columns):
 #   Column                              Non-Null Count  Dtype  
---  ------                              --------------  -----  
 0   search_id                           26500 non-null  object 
 1   user_id                             26500 non-null  object 
 2   search_query                        26500 non-null  object 
 3   search_date                         26500 non-null  object 
 4   results_returned                    26500 non-null  int64  
 5   clicked_result_position             26500 non-null  int64  
 6   device_type                         26500 non-null  object 
 7   search_duration_seconds             26500 non-null  float64
 8   had_typo                            26500 non-null  bool   
 9   used_filters                        26500 non-null  bool   
 10  location_country                    26500 non-null  object 
 11  is_clicked_result_position_missing  26500

## 5. Formating data types for TSQL 

In [51]:
df_silver['search_date'] = pd.to_datetime(df_silver['search_date'], format='%Y-%m-%d')
df_silver['search_date'] = pd.to_datetime(df_silver['search_date'], errors='coerce')
df_silver['search_date'] = df_silver['search_date'].dt.strftime('%Y-%m-%d')

df_silver['search_date'].info()

<class 'pandas.core.series.Series'>
RangeIndex: 26500 entries, 0 to 26499
Series name: search_date
Non-Null Count  Dtype 
--------------  ----- 
26500 non-null  object
dtypes: object(1)
memory usage: 207.2+ KB


## 6. Saving final df_silver in csv

In [52]:
output_file = 'netflix_silver_layer_search_logs.csv'

df_silver.to_csv(
    output_file,
    index = False,
    sep = ','
)